# NLP Triage Kiosk: Local Testing
This notebook lets you test the ASR, VAD, Acoustic Distress, and Negation Logic using your own text or audio.

## 1. Setup and Installation
Run this cell once to install the dependencies and the `spaCy` NLP model.

In [ ]:
!pip install -r ../requirements_nlp.txt
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1.tar.gz

## 2. Initialize the Analyzer
Constructing `TriageKioskAnalyzer()` no longer loads any model eagerly. PyTorch (Silero VAD) and HuggingFace (Whisper base, ~140MB) models are downloaded/loaded lazily on first use of the audio methods below (`detect_voice_activity`, `transcribe_audio`, ...). Text-only methods (negation/complaint/identity) do not need this class or these downloads at all — see the next cell.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

from resilicare.nlp import TriageKioskAnalyzer
analyzer = TriageKioskAnalyzer()  # cheap: no models loaded yet

## 3. Test Text Processing (Negation & Entity Extraction)
These are dependency-free module-level functions — they work without torch/librosa/transformers/spaCy installed (spaCy is used only if present; otherwise identity binding falls back to an ephemeral alias). `process_kiosk_text` is also the manual-fallback entry point used by the demo server's `/api/kiosk/text` route.

In [ ]:
from resilicare.nlp.text_pipeline import (
    detect_acuity_with_negation, extract_chief_complaint, patient_identity_binding, process_kiosk_text,
)

test_text = "I am not bleeding profusely, but my chest hurts severely. My name is John."

print("Input Text:", test_text)
print("
--- Analysis ---")
print("Red Flags (Negation Check):", detect_acuity_with_negation(test_text))
print("Chief Complaint (canonical trigger phrase):", extract_chief_complaint(test_text))
print("Patient Identity:", patient_identity_binding(test_text))
print("
Full text pipeline (process_kiosk_text):", process_kiosk_text(test_text))

## 4. Test with Real Audio (Full Pipeline)
Record a short `.wav` file (e.g., using your phone or a voice recorder app) and save it as `test.wav` in this `examples/` directory. Then, uncomment and run the cell below to test the full pipeline (VAD -> Acoustic Distress -> Whisper ASR -> Extraction).

In [ ]:
import json

# Ensure you have a 'test.wav' file in this directory
audio_file = 'test.wav'

if os.path.exists(audio_file):
    print(f"Processing {audio_file}...")
    result = analyzer.process_kiosk_interaction(audio_file)
    print(json.dumps(result, indent=2))
else:
    print(f"Please record a {audio_file} file to test the audio pipeline.")